# 🧭 option_signals — do option prices tell which stocks will do better?

**Research only.** Option traders sometimes know things before stock traders do, and it shows in
option prices. Four published signals read that from the chains; this notebook tests whether they
rank stocks by their coming return, with the market taken out, before anything is traded. The test
is pre-registered in `README.md`.

1. **Data**: daily bars, dividends, and implied volatilities computed from the end-of-day chains.
2. **Data checks**: are the volatilities sensible, is the universe the right size, look-ahead audit.
3. **The signals**: how they are spread and how they relate to each other.
4. **The test**: rank correlation (IC) of each signal with the next 1 / 5 / 21 sessions' market-neutral
   return, a partial IC after common controls, month by month, and a simple long-short book.
5. **Pictures**.
6. **Verdict**: `CHECKS` table and one `SCORECARD` line.
7. **Holdout**: locked behind `READ_HOLDOUT`; read once, only for cells that pass.

The first run builds caches in `var/option_signals/` (the chains take ~15 min); later runs take a
minute or two.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel (README "Quick start")
import importlib.util
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import plots, strategies

_spec = importlib.util.spec_from_file_location(
    "option_signals_research", strategies.STRATEGIES_DIR / "option_signals" / "research.py")
os_rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(os_rs)

# --- the pre-registered windows (README) ---------------------------------------------------
DAILY_START = "2024-09-01"     # a year of bars before the chains: dividends, betas
DISC = dict(months=("2025-10", "2026-04"), first_t="2025-10-20", daily_end="2026-04-30")
HOLD = dict(months=("2026-05", "2026-08"), first_t="2026-05-01", last_t="2026-08-31",
            daily_end="2026-10-16")
READ_HOLDOUT = False           # True reads the locked window ONCE, for discovery-passing cells only
HOLD_ALPHA = 0.05              # holdout bar, divided by the number of cells taken there
pd.set_option("display.width", 170)

## 1 · Data

- **Daily bars** (shared DB, from 2024-09): adjusted open / close for returns, raw close and volume.
- **Dividends**: every ex-date and its size, read from the steps of adjusted ÷ raw close; the next
  one is projected from the last four (README).
- **Surfaces** (`os_rs.surfaces`): for every name and day, expiry A (21–60 days, nearest 30) and the
  Black-Scholes implied volatility of each quoted contract near the money, reduced to the ATM call
  and put, the 0.95 put and the open-interest-weighted call − put IV spread; plus the day's call and
  put volume.

The DB load ends on the window's last day, so nothing after it can leak in.

In [ ]:
def load(months, daily_end, extra=None):
    symbols = sorted(p.name for p in os_rs.CHAINS.iterdir() if p.is_dir())
    daily = os_rs.load_daily(symbols, DAILY_START, daily_end)
    sessions = pd.DatetimeIndex(sorted(daily["date"].unique()))
    cols = sorted(daily["symbol"].unique())
    px = {c: os_rs.wide(daily, c, sessions, cols) for c in ("open", "close", "adj_close", "volume")}
    events = os_rs.dividend_events(px["close"], px["adj_close"])
    surf = os_rs.surfaces(*months, events, symbols)
    if extra is not None:
        surf = pd.concat([extra, surf], ignore_index=True)
    sw = {c: os_rs.wide(surf, c, sessions, cols) for c in
          ("S", "iv_call", "iv_put", "iv_otm_put", "cpiv", "call_vol", "put_vol", "div")}
    adj = px["adj_close"]
    open_adj = px["open"] * adj / px["close"]
    dvol = px["close"] * px["volume"]
    sig = os_rs.signals(sw, adj, px["volume"])
    mask = os_rs.universe(sig["iv_atm"], sw["S"], dvol)
    beta, mkt = os_rs.market_beta(adj)
    controls = [np.log(dvol.rolling(20, min_periods=10).median().replace(0, np.nan)),
                adj / adj.shift(5) - 1,
                adj / adj.shift(21) - 1, sig["rv21"]]
    return dict(daily=daily, sessions=sessions, px=px, events=events, surf=surf, sw=sw, adj=adj,
                open_adj=open_adj, dvol=dvol, sig=sig, mask=mask, beta=beta, controls=controls,
                med_dvol=dvol.rolling(20, min_periods=10).median())


D = load(DISC["months"], DISC["daily_end"])
win = D["sessions"] >= pd.Timestamp(DISC["first_t"])
MASK = D["mask"].mul(pd.Series(win, index=D["sessions"]), axis=0).astype(bool)
surf = D["surf"]
print(f"daily bars: {D['daily']['symbol'].nunique():,} names, {len(D['sessions'])} sessions "
      f"{D['sessions'][0]:%Y-%m-%d} → {D['sessions'][-1]:%Y-%m-%d}")
print(f"dividend payers: {len(D['events']):,}; surfaces: {len(surf):,} name-days, "
      f"{surf['symbol'].nunique():,} names, "
      f"{surf['date'].min():%Y-%m-%d} → {surf['date'].max():%Y-%m-%d}")
print(f"universe: {int(MASK.sum(axis=1)[win].median())} names per day (median), "
      f"{int(win.sum())} signal days from {DISC['first_t']}")

## 2 · Data checks

- **IVs make sense**: the ATM IV should track each name's realized volatility across names, the ATM
  call and put IVs should agree (after dividends), and puts below the money should be dearer (skew > 0).
- **Coverage**: how many universe names carry each signal.
- **Look-ahead audit**: every signal recomputed on data cut at a date is unchanged before the cut.
- **Timing**: the forward return starts at t+1's open, so no signal can see its own outcome.

In [ ]:
S = D["sig"]
on = MASK
iv = S["iv_atm"].where(on).stack()
rv = S["rv21"].where(on).stack()
gap = (D["sw"]["iv_call"] - D["sw"]["iv_put"]).where(on).stack()
cs = pd.DataFrame({"iv": S["iv_atm"].where(on).median(),
                   "rv": S["rv21"].where(on).median()}).dropna()
cover = {k: float(S[k].where(on).notna().sum().sum() / on.sum().sum()) for k in os_rs.SIGNALS}
print(f"ATM IV median {iv.median():.1%} (p10 {iv.quantile(.1):.1%}, p90 {iv.quantile(.9):.1%}); "
      f"realized 21d median {rv.median():.1%}")
print(f"cross-name correlation of median IV and median realized vol: {cs['iv'].corr(cs['rv']):.2f}")
print(f"ATM call − put IV: median {gap.median():+.4f}, "
      f"|gap| > 5 points: {(gap.abs() > 0.05).mean():.1%}")
print("median signal values in the universe:",
      {k: round(float(S[k].where(on).stack().median()), 4) for k in os_rs.SIGNALS})
print("coverage of universe name-days:", {k: round(v, 3) for k, v in cover.items()})

cut = D["sessions"][D["sessions"].searchsorted(pd.Timestamp("2026-02-02"))]
sw_cut = {k: v.loc[:cut] for k, v in D["sw"].items()}
trunc = os_rs.signals(sw_cut, D["adj"].loc[:cut], D["px"]["volume"].loc[:cut])
causal = all(S[k].loc[:cut].equals(trunc[k]) for k in os_rs.SIGNALS)
everyone = pd.DataFrame(True, index=D["sessions"], columns=MASK.columns)
raw1 = os_rs.residual_forward(D["open_adj"], D["beta"] * 0, everyone, 1)   # beta 0: the raw return
expect = D["open_adj"].shift(-2) / D["open_adj"].shift(-1) - 1
timing = bool(np.allclose(raw1.stack(), expect.stack().reindex(raw1.stack().index)))
print(f"look-ahead audit (cut {cut:%Y-%m-%d}): signals identical before the cut = {causal}")
print(f"forward return = open(t+1) → open(t+2) for h = 1: {timing}")

## 3 · The signals

How each signal is spread across the universe, and how alike they are: the mean over days of the
cross-sectional rank correlation between two signals. `vrp` and the ATM IV level, and `skew` and
`cpiv`, are related by construction, so some overlap is expected.

In [ ]:
desc = pd.DataFrame({k: S[k].where(on).stack().describe(percentiles=[.1, .5, .9])
                     for k in os_rs.SIGNALS}).T
print(desc[["count", "10%", "50%", "90%"]].round(4).to_string())
names = list(os_rs.SIGNALS) + ["iv_atm", "rv21"]
ranks = {k: S[k].where(on).rank(axis=1) for k in names}
corr = pd.DataFrame({a: {b: float(ranks[a].T.corrwith(ranks[b].T).mean()) for b in names}
                     for a in names})
print("\nmean daily rank correlation:")
print(corr.round(2).to_string())

## 4 · The test (discovery window)

For each signal and horizon h (sessions): `ic` = the mean daily rank correlation between the signal
and the market-neutral forward return, `t` its Newey-West t-statistic (lag h); `pic` / `pt` the same
after removing size, the 5- and 21-day return and the 21-day volatility from the signal; `month_share`
the share of months where the IC had the published sign; `mde` the smallest IC this window could
find 80% of the time at the bar. `ic_t1..t3`: IC in the smallest / middle / largest third by dollar
volume. `book_*`: a long-short quintile book rebalanced every h sessions, annualized, net = after
5 bp per trade at its turnover.

**Pass** = IC with the published sign at t ≥ 2.65, partial IC with the same sign at t ≥ 2.0, and the
published sign in ≥ 60% of the months. Expected signs: skew −, cpiv +, vrp −, os −.

In [ ]:
FWD = {h: os_rs.residual_forward(D["open_adj"], D["beta"], MASK, h) for h in os_rs.HORIZONS}
R = {}
for k, sign in os_rs.SIGNALS.items():
    for h in os_rs.HORIZONS:
        R[(k, h)] = os_rs.score_signal(S[k], FWD[h], MASK, D["controls"], D["med_dvol"], sign, h)
cols = ["days", "ic", "t", "pic", "pt", "months", "month_share", "mde", "ic_t1", "ic_t2", "ic_t3",
        "book_gross", "book_net", "book_turnover", "book_hit", "passed"]
T = pd.DataFrame({key: {c: r[c] for c in cols} for key, r in R.items()}).T
T.index.names = ["signal", "h"]
T.insert(0, "sign", [os_rs.SIGNALS[k] for k, _ in T.index])
print(T.to_string(float_format=lambda x: f"{x:.4f}"))

**IC by month** (h = 5): a real effect should show its sign in most months, not in one.

In [ ]:
M = pd.DataFrame({k: R[(k, 5)]["ic_series"].groupby(lambda d: d.to_period("M")).mean()
                  for k in os_rs.SIGNALS}).dropna(how="all")
print(M.round(4).to_string())

**Exploratory (NOT part of the verdict).** Three variants the same papers also report, scored the
same way, to see whether the level signals simply hid a working form: `cpiv_chg5` (the call − put
spread minus its value 5 sessions earlier; Cremers & Weinbaum), `ivgap_chg21` (the 21-session change
of ATM call IV minus that of ATM put IV; An, Ang, Bali & Cakici 2014; expected +), and `composite`
(the mean of the four signals' daily cross-sectional z-scores, each flipped to its published sign).
A strong cell here would only be a hypothesis for its own pre-registered holdout test.

In [ ]:
def z(x):
    return x.sub(x.mean(axis=1), axis=0).div(x.std(axis=1), axis=0)


sw = D["sw"]
EXTRA = {"cpiv_chg5": (S["cpiv"] - S["cpiv"].shift(5), 1),
         "ivgap_chg21": ((sw["iv_call"] - sw["iv_call"].shift(21))
                         - (sw["iv_put"] - sw["iv_put"].shift(21)), 1),
         "composite": (sum(sign * z(S[k].where(MASK)) for k, sign in os_rs.SIGNALS.items()) / 4,
                       1)}
RX = {(k, h): os_rs.score_signal(sig, FWD[h], MASK, D["controls"], D["med_dvol"], sign, h)
      for k, (sig, sign) in EXTRA.items() for h in os_rs.HORIZONS}
TX = pd.DataFrame({key: {c: r[c] for c in ["days", "ic", "t", "pic", "pt", "month_share", "mde",
                                           "book_gross", "book_net"]}
                   for key, r in RX.items()}).T
print(TX.to_string(float_format=lambda x: f"{x:.4f}"))

## 5 · Pictures

**Left**: each signal's cumulative IC at h = 5, flipped to its published sign (a line that climbs
steadily = the signal keeps working; flat or falling = it does not). **Right**: each cell's t against
the pass bar.

In [ ]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.1,
                    subplot_titles=("cumulative IC at h = 5, published sign", "t of the mean IC"))
for i, (k, sign) in enumerate(os_rs.SIGNALS.items()):
    c = (sign * R[(k, 5)]["ic_series"]).dropna().cumsum()
    fig.add_trace(plots._scatter(c, name=k, line={"color": plots.SERIES[i], "width": 2},
                                 hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}<extra>" + k + "</extra>"),
                  1, 1)
    ts = [sign * R[(k, h)]["t"] for h in os_rs.HORIZONS]
    fig.add_trace(go.Bar(x=[f"h={h}" for h in os_rs.HORIZONS], y=ts, name=k, showlegend=False,
                         marker={"color": plots.SERIES[i]},
                         hovertemplate="%{x}<br>t %{y:.2f}<extra>" + k + "</extra>"), 1, 2)
fig.add_hline(y=0, line={"color": plots._INK, "width": 1}, row=1, col=1)
fig.add_hline(y=os_rs.T_BAR, line={"color": plots._MUTED, "dash": "dash"}, row=1, col=2)
fig.add_hline(y=0, line={"color": plots._INK, "width": 1}, row=1, col=2)
fig.update_xaxes(type="date", row=1, col=1)
plots._base(fig, title="Do option signals rank stocks? (discovery window)", height=460)
fig.update_layout(barmode="group", legend={"orientation": "h", "yanchor": "top", "y": -0.15,
                                           "x": 0, "xanchor": "left"}, margin={"b": 90})
fig.show()

## 6 · Verdict

In [ ]:
CHECKS = pd.DataFrame([
    ("ATM IV tracks realized vol across names (corr ≥ 0.7)", f"{cs['iv'].corr(cs['rv']):.2f}",
     cs["iv"].corr(cs["rv"]) >= 0.7),
    ("ATM call − put IV: median within ±1 point", f"{gap.median():+.4f}",
     abs(gap.median()) <= 0.01),
    ("puts below the money dearer: median skew > 0",
     f"{S['skew'].where(on).stack().median():+.4f}", S["skew"].where(on).stack().median() > 0),
    ("universe ≈ 500 names a day", f"{int(MASK.sum(axis=1)[win].median())}",
     abs(MASK.sum(axis=1)[win].median() - 500) <= 25),
    ("signals unchanged by cutting later data", str(causal), causal),
    ("forward return starts at t+1's open", str(timing), timing),
], columns=["check", "value", "ok"])
print(CHECKS.to_string(index=False))
passing = [key for key, r in R.items() if r["passed"]]
print(f"\n{len(passing)} of {len(R)} cells pass discovery: {passing or 'none'}"
      + ("" if passing else " — the holdout stays unread."))
SCORECARD = {"notebook": "option_signals", "window": "discovery",
             "checks_ok": bool(CHECKS["ok"].all()),
             "cells": {f"{k}/h{h}": {"ic": round(r["ic"], 5), "t": round(r["t"], 2),
                                     "pt": round(r["pt"], 2), "share": round(r["month_share"], 2),
                                     "mde": round(r["mde"], 4), "passed": r["passed"]}
                       for (k, h), r in R.items()},
             "passing": [f"{k}/h{h}" for k, h in passing]}
print("SCORECARD " + json.dumps(SCORECARD))

## 7 · Holdout (locked)

Signal days 2026-05-01 → 2026-08-31, scored ONCE, only for cells that passed discovery, each at
one-sided p ≤ 0.05 ÷ (number of such cells) with the published sign. The chains for May–August are
read only here; the April chains keep the 5-session option volume warm.

In [ ]:
from scipy.stats import norm

if not passing:
    print("Nothing passed discovery: the holdout stays unread.")
elif not READ_HOLDOUT:
    print(f"Holdout locked (READ_HOLDOUT = False); it would score {passing}.")
else:
    DH = load(HOLD["months"], HOLD["daily_end"], extra=surf[surf["date"] >= "2026-04-01"])
    hw = (DH["sessions"] >= pd.Timestamp(HOLD["first_t"])) & \
        (DH["sessions"] <= pd.Timestamp(HOLD["last_t"]))
    HM = DH["mask"].mul(pd.Series(hw, index=DH["sessions"]), axis=0).astype(bool)
    alpha = HOLD_ALPHA / len(passing)
    rows = {}
    for k, h in passing:
        fwd = os_rs.residual_forward(DH["open_adj"], DH["beta"], HM, h)
        r = os_rs.score_signal(DH["sig"][k], fwd, HM, DH["controls"], DH["med_dvol"],
                               os_rs.SIGNALS[k], h)
        p = float(norm.sf(os_rs.SIGNALS[k] * r["t"]))
        rows[(k, h)] = {**{c: r[c] for c in cols if c != "passed"}, "p": p,
                        "holdout pass": p <= alpha}
    TH = pd.DataFrame(rows).T
    print(f"holdout bar: published sign with one-sided p ≤ {alpha:.4f}")
    print(TH.to_string(float_format=lambda x: f"{x:.4f}"))
    print("SCORECARD " + json.dumps({"notebook": "option_signals", "window": "holdout",
                                     "cells": {f"{k}/h{h}": {"ic": round(v["ic"], 5),
                                                             "p": round(v["p"], 4),
                                                             "pass": bool(v["holdout pass"])}
                                               for (k, h), v in rows.items()}}))